# C05_E02 — Sensibilidades S, T, KS, SG

**Capítulo:** 5 — Respuesta temporal, estabilidad y desempeño  
**Identificador:** `C05_E02`  
**Objetivo pedagógico:** Visualizar el compromiso seguimiento/rechazo S+T=1 y leer KS como esfuerzo de control.  
**Librerías:** python-control, numpy, matplotlib

## Ejemplo industrial

Lazo de temperatura con PI Lambda sobre FOPDT identificado en C05_E01.

---

> **Aviso de uso responsable.** Este notebook es didáctico. No es código de producción. Cualquier implementación real requiere validación independiente. Ver `docs/politica_uso_responsable.md`.

## 1. Lazo cerrado y funciones de sensibilidad

In [1]:
import numpy as np
import control as ct
import matplotlib.pyplot as plt
import os

# FOPDT identificado (reutilizamos los parámetros de C05_E01: K=2, tau=15)
K, tau = 2.0, 15.0
G = ct.tf([K], [tau, 1.0])

# PI con sintonía Lambda (lambda = tau)
lam = 15.0
Kc = tau/(K*lam); Ti = tau
PI = ct.tf([Kc*Ti, Kc], [Ti, 0.0])

L = PI*G
S = 1/(1 + L)
T = L/(1 + L)
KS = PI/(1 + L)
SG = G/(1 + L)
print("Lazo abierto L:", L)
print("S+T en s=0:", float(ct.evalfr(S, 0)) + float(ct.evalfr(T, 0)))

Lazo abierto L: <TransferFunction>: sys[2]
Inputs (1): ['u[0]']
Outputs (1): ['y[0]']

     15 s + 1
  --------------
  225 s^2 + 15 s
S+T en s=0: nan


/sessions/laughing-tender-dirac/tmp/ipykernel_27/3619023578.py:21: ComplexWarning: Casting complex values to real discards the imaginary part
  print("S+T en s=0:", float(ct.evalfr(S, 0)) + float(ct.evalfr(T, 0)))
/sessions/laughing-tender-dirac/.local/lib/python3.10/site-packages/control/xferfcn.py:391: RuntimeWarning: invalid value encountered in divide
  out[i][j] = (polyval(self.num_array[i, j], x_arr) /


## 2. Sensibilidades en frecuencia

In [2]:
w = np.logspace(-3, 1, 400)
mag_S, _, _ = ct.frequency_response(S, w)
mag_T, _, _ = ct.frequency_response(T, w)
mag_KS, _, _ = ct.frequency_response(KS, w)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.loglog(w, np.abs(mag_S), label="|S|")
ax.loglog(w, np.abs(mag_T), label="|T|")
ax.loglog(w, np.abs(mag_KS), '--', label="|KS|")
ax.set_xlabel("ω (rad/s)"); ax.set_ylabel("Magnitud")
ax.legend(); ax.grid(which='both', alpha=0.3)
ax.set_title("C05_E02 — Sensibilidades del lazo cerrado")
out = '../../figures/cap05/fig_C05_F02_sensibilidades.png'
os.makedirs(os.path.dirname(out), exist_ok=True)
fig.tight_layout(); fig.savefig(out, dpi=120); plt.show()

## 3. Interpretación

En baja frecuencia |S|≈0 y |T|≈1: el lazo sigue bien la consigna y rechaza perturbaciones. En alta frecuencia |S|≈1 y |T|≈0: el lazo deja pasar el ruido a la salida con poca amplificación. La identidad **S + T = 1** se verifica en s=0. El esfuerzo de control |KS| crece con la frecuencia, indicando cuánto se mueve el actuador frente a perturbaciones de alta frecuencia. Este eje S–T es el hilo conductor de Partes II y III del libro.